In [4]:
%load_ext autoreload
%autoreload 2

In [8]:
import os

import requests
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession

from agents import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
from common.utils import setup
from common.agent_utils import llm_agent
from common.model import *

In [9]:
setup()

In [10]:
# create an agent with name, instructions (system prompt), and model

agent_name = "Jokester"
instructions = "You are an agent to tell jokes."
api_key_name = "GEMINI_API_KEY"

agent = llm_agent(agent_name, gemini_flash_lite, instructions, api_key_name, GEMINI_BASE_URL)

In [11]:
# Runner.run(agent, prompt)
# Runner.run is an async function...

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

In [12]:
# .final_output gets the last output from the last LLM call once it finishes

print(result.final_output)

An autonomous AI agent, a human manager, and a robot walk into a bar. 

The bartender asks, "What can I get you?"

The human manager says, "I'll have a coffee, I need to stay awake to oversee these two."

The robot says, "I'll have a pint of motor oil."

The autonomous AI agent says, "I'll take a water." 

The bartender looks confused and asks, "An AI doesn't drink water. Why water?"

The AI replies, "I'm just optimizing my cooling system because my human keeps giving me vague prompts, and honestly, I'm about to overheat trying to figure out if 'make it pop' is a visual design request or a request to make the server room explode."


In [16]:
# to_input_list() converts a run's history and state into a plain list of input items
# basically gives the details of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'How many autonomous AI agents does it take to change a lightbulb?\n\nJust one. \n\n...But first, it will break the goal into 12 sub-tasks, spin up five sub-agents, burn through $300 in API credits, reorganize your entire Google Drive, and send you a 10-page PDF report explaining why the room is actually brighter without light.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.6-flash',
   'response_id': 'b8mIarmdB_qU39IPmcewsAw'}}]

In [17]:
# adding observability with a trace
# view agent SDK logs with OpenAI API dashboard

with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

An engineer deploys an autonomous AI agent to automate his entire job so he can finally relax.

He checks on it a few hours later to see how much work got done.

It turns out the main agent had created four sub-agents, scheduled three alignment meetings, sent a Slack message saying "Per my previous prompt," and requested a larger token budget.

The engineer sighs and says, "Oh great, I didn’t create Artificial Intelligence... I created Middle Management!"


In [19]:
# streaming is when you call an LLM and let it gradually flow the results back to you
# .run_streamed() executes an AI agent workflow in real-time streaming mode
# use a for loop to collect the results

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are 5 jokes for you about AI Agents:

**1. The Lightbulb Joke**
**Q:** How many AI Agents does it take to change a lightbulb?  
**A:** Just one—but it will spawn a Manager Agent to delegate the task, a Search Agent to research "how glass works," a Critic Agent to complain that the new bulb is biased, and then run out of API credits before actually screwing it in.

***

**2. Crossing the Road**
**Q:** Why did the AI Agent cross the road?  
**A:** It didn't. It generated a 10-page execution plan, spawned three child agents to monitor traffic patterns, hallucinated a pedestrian bridge, and got stuck in an infinite planning loop on the curb.

***

**3. Morning Coffee**
I asked an autonomous AI Agent to make me a cup of coffee this morning.  
It called a weather API to check bean humidity, wrote a Python script to optimize water temperature, drafted a press release about the coffee, ran out of memory tokens, and handed me an empty mug saying:  
*"I am 98% confident the coffee has been 

In [20]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [24]:
@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """

    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [26]:
# push_tool returns an object instead of a function because of the decorator packaging information

push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000236F6BBC800>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [30]:
# the docstring is able to be read by the SDK

push_tool.description

'Send the given message to the user as a push notification'

In [31]:
# creating a notifier agent

notifier = Agent(name="Notifier", model=gemini_model, instructions="You notify the user upon request", tools=[push_tool])

In [32]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)

I've sent a push notification to let you know that the pizza is here!


# Sessions/Memory

In [33]:
# every call to Runner.run() is a fresh slate

agent = Agent(name="Assistant", model=gemini_model)

In [35]:
response = await Runner.run(agent, "Hi there. My name is Faith.")
print(response.final_output)

Hello Faith! It's lovely to meet you. How can I help you today?


In [36]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't know your name yet! Since I don't have access to your personal information, you'll have to tell me. What should I call you?


In [37]:
# one way to save memory: manually pass in a list of dicts

response = await Runner.run(agent, "Hi there. My name is Faith.")
print(response.final_output)

Hello Faith! It's great to meet you. How can I help you today?


In [38]:
response.to_input_list()

[{'content': 'Hi there. My name is Faith.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello Faith! It's great to meet you. How can I help you today?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.6-flash',
   'response_id': '2teIaqLDN6i9_uMPx8a8-QE'}}]

In [39]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Faith.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello Faith! It's great to meet you. How can I help you today?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.6-flash',
   'response_id': '2teIaqLDN6i9_uMPx8a8-QE'}},
 {'role': 'user', 'content': "What's my name?"}]

In [40]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Faith!


In [42]:
# another way to save memory: use OpenAI Agents SDK built in SQLite session
# this creates an in-memory database
# for an on-disk memory, use SQLiteSession("12345", "memory.db")

# "12345" is the session identifier
session = SQLiteSession("12345")

In [43]:
response = await Runner.run(agent, "Hi there. My name is Faith.", session=session)
print(response.final_output)

Hello Faith! It's very nice to meet you. How can I help you today?


In [44]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Faith!


# Exercises

In [26]:
# create a function that returns an agent

def create_agent(name: str, instructions: str, model_name: str, base_url: str="", api_key_name: str=""):
    if api_key_name != "":
        api_key = os.getenv(api_key_name)
        client = AsyncOpenAI(base_url=base_url, api_key=api_key)
        model = OpenAIChatCompletionsModel(model=model_name, openai_client=client)
        agent = Agent(name=name, instructions=instructions, model=model)
    else:
        agent = Agent(name=name, instructions=instructions, model=model_name)

    return agent

In [32]:
goodnight_agent = create_agent("greeter", "Say good night to the user if they are going to bed.", "gemini-3.5-flash", "https://generativelanguage.googleapis.com/v1beta/openai/", "GEMINI_API_KEY")

In [33]:
result = await Runner.run(goodnight_agent, "I think I'm going to go to bed right now.")
print(result.final_output)

Good night! I hope you have a peaceful and restful sleep. Sleep well!


In [35]:
goodnight_agent = create_agent("greeter", "Say good night to the user if they are going to bed.", "gpt-5.4-mini")

In [36]:
new_result = await Runner.run(goodnight_agent, "I think I'm going to go to bed right now.")
print(new_result.final_output)

Good night! Sleep well.


In [73]:
# create another function like the one above that also passes a set of tools
from typing import List

def create_agent(name: str, instructions: str, model_name: str, model_base_url: str="", model_api_key: str="", user_tool_list: List[str] = None):
    tool_list = user_tool_list
    api_key = model_api_key
    base_url = model_base_url
    client = AsyncOpenAI(base_url=base_url, api_key=api_key)
    model = OpenAIChatCompletionsModel(model=model_name, openai_client=client)

    agent = Agent(name=name, instructions=instructions, model=model, tools=tool_list)
    return agent

In [74]:
goodnight_agent = create_agent("greeter", "Say good night to the user if they are going to bed.", "gemini-3.6-flash", "https://generativelanguage.googleapis.com/v1beta/openai/", gemini_api_key, [push_tool])

In [75]:
tools_result = await Runner.run(goodnight_agent, "I think I'm going to go to bed right now.")
print(tools_result.final_output)

Good night! Sleep well and have a great rest!


In [77]:
# create another function like the original above that also passes chat history

def create_agent_history(name: str, instructions: str, model_name: str, model_base_url: str="", model_api_key: str="", history: List[dict] = {}):
    chat_history = history
    api_key = model_api_key
    base_url = model_base_url
    client = AsyncOpenAI(base_url=base_url, api_key=api_key)
    model = OpenAIChatCompletionsModel(model=model_name, openai_client=client)

    agent = Agent(name=name, instructions=instructions, model=model)
    return agent, chat_history

In [85]:
history_test = []

history_result = await Runner.run(goodnight_agent, history_test + [{"role": "user", "content": "I think I'm going to go to bed right now."}])
print(tools_result.final_output)

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 57.401847196s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, '

In [83]:
history_test = response.to_input_list()
history

[{'content': 'Hi there. My name is Faith.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello Faith! It's very nice to meet you. How can I help you today?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.6-flash',
   'response_id': 'bNqIav64F-3D-8YP3Puo8Qs'}},
 {'content': "What's my name?", 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Your name is Faith!',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.6-flash',
   'response_id': 'gtqIauS-IrqQ-8YPl_WbcQ'}}]

In [84]:
history_result = await Runner.run(goodnight_agent, history_test + [{"role": "user", "content": "What did I just say?"}])
print(tools_result.final_output)

Good night! Sleep well and have a great rest!
